In [1]:
import os
import random
os.environ['PYTHONHASHSEED'] = '42'

import pandas as pd
import numpy as np
np.random.seed(42)
random.seed(42)

import matplotlib.pyplot as plt
import seaborn as sns

# Thư viện tiền xử lý và mô hình
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, f1_score, roc_auc_score, recall_score
from sklearn.ensemble import RandomForestClassifier

# Các thuật toán yêu cầu
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from imblearn.over_sampling import SMOTE

import tensorflow as tf
tf.random.set_seed(42)

# Tắt cảnh báo để giao diện sạch sẽ
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Đọc dữ liệu (Đảm bảo bạn đã upload file lên Colab hoặc cùng thư mục)
df = pd.read_csv('healthcare-dataset-stroke-data.csv')

In [3]:
# 1. Xử lý giá trị thiếu cho BMI
df['bmi'] = df['bmi'].fillna(df['bmi'].median())

In [4]:
# 2. Loại bỏ cột ID
df.drop(['id'], axis=1, inplace=True)

In [5]:
# 3. Mã hóa biến phân loại (Categorical to Numerical)
# Chuyển đổi các cột chữ thành các cột số 0 và 1
df = pd.get_dummies(df, columns=['gender', 'ever_married', 'work_type', 'Residence_type', 'smoking_status'], drop_first=True)

In [6]:
# 4. Tách đặc trưng (X) và nhãn (y)
X = df.drop('stroke', axis=1)
y = df['stroke']

In [7]:
# 5. Chia tập Train/Test
# Train: 70%
# Validation: 15%
# Test: 15%

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

In [8]:
# 6. Xử lý mất cân bằng bằng SMOTE (Tạo thêm dữ liệu cho nhóm thiểu số)
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

In [9]:
# 7. Chuẩn hóa dữ liệu (Bắt buộc cho SVM và Neural Network)
scaler = StandardScaler()
X_train_res = scaler.fit_transform(X_train_res)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Kích thước tập huấn luyện sau khi SMOTE: {X_train_res.shape}")

Kích thước tập huấn luyện sau khi SMOTE: (6806, 16)


In [10]:
def fitness_function(individual, X_tr, y_tr, X_te, y_te):
    selected = [i for i, bit in enumerate(individual) if bit == 1]
    if len(selected) == 0: return 0
    # Dùng Decision Tree để đánh giá nhanh "độ tốt" của bộ đặc trưng
    model = DecisionTreeClassifier(max_depth=5, random_state=42)
    model.fit(X_tr[:, selected], y_tr)
    preds = model.predict(X_te[:, selected])

    f1 = f1_score(y_te, preds)

    # Penalty for too many features
    penalty = len(selected) / X_tr.shape[1]

    # fitness = 0.97 * f1 - 0.03 * penalty
    fitness = f1 - 0.02 * penalty

    return fitness

def run_ga(X_tr, y_tr, X_te, y_te, n_features):
    # Thiết lập seed trước GA để đảm bảo kết quả không thay đổi
    np.random.seed(42)
    
    pop_size = 20
    generations = 15
    mutation_rate = 0.03
    # Khởi tạo quần thể ngẫu nhiên
    population = np.random.randint(0, 2, (pop_size, n_features))
    
    for g in range(generations):
        scores = [fitness_function(ind, X_tr, y_tr, X_te, y_te) for ind in population]
        # Chọn lọc các cá thể tốt nhất
        best_indices = np.argsort(scores)[-pop_size//2:]
        parents = population[best_indices]
        
        # Tạo thế hệ mới qua lai ghép
        offspring = []
        for i in range(pop_size - len(parents)):
            p1, p2 = parents[np.random.randint(0, len(parents), 2)]
            child = np.where(np.random.rand(n_features) > 0.5, p1, p2)
            # Mutation
            for j in range(n_features):
                if np.random.rand() < mutation_rate:
                    child[j] = 1 - child[j]
            offspring.append(child)
        
        population = np.vstack([parents, offspring])
        print(f"Gen {g+1}: Best F1 = {max(scores):.4f}")

    final_scores = [
        fitness_function(ind, X_tr, y_tr, X_te, y_te)
        for ind in population
    ]

    best_individual = population[np.argmax(final_scores)]
    
    return best_individual

# Chạy GA để chọn đặc trưng
best_mask = run_ga(X_train_res, y_train_res, X_val, y_val, X_train_res.shape[1])
selected_features_idx = [i for i, bit in enumerate(best_mask) if bit == 1]

selected_feature_names = X.columns[selected_features_idx].tolist()
print(f"Các đặc trưng được chọn: {selected_feature_names}")
print(f"Số lượng đặc trưng tối ưu được chọn: {len(selected_features_idx)}")

Gen 1: Best F1 = 0.2400
Gen 2: Best F1 = 0.2400
Gen 3: Best F1 = 0.2400
Gen 4: Best F1 = 0.2546
Gen 5: Best F1 = 0.2546
Gen 6: Best F1 = 0.2546
Gen 7: Best F1 = 0.2546
Gen 8: Best F1 = 0.2546
Gen 9: Best F1 = 0.2546
Gen 10: Best F1 = 0.2546
Gen 11: Best F1 = 0.2546
Gen 12: Best F1 = 0.2547
Gen 13: Best F1 = 0.2547
Gen 14: Best F1 = 0.2547
Gen 15: Best F1 = 0.2547
Các đặc trưng được chọn: ['age', 'avg_glucose_level', 'gender_Male', 'gender_Other', 'work_type_Private', 'Residence_type_Urban', 'smoking_status_smokes']
Số lượng đặc trưng tối ưu được chọn: 7


In [11]:
# Lọc lại dữ liệu theo GA
X_train_ga = X_train_res[:, selected_features_idx]
X_test_ga = X_test[:, selected_features_idx]
X_val_ga = X_val[:, selected_features_idx]

In [12]:
models = {
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42),
    "SVM": SVC(kernel='rbf', probability=True, C=10, gamma=0.01, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_ga, y_train_res)
    # y_pred = model.predict(X_test_ga)
    y_probs = model.predict_proba(X_test_ga)[:, 1]

    threshold = 0.45

    y_pred = (y_probs > threshold).astype(int)
    print(f"\n--- {name} ---")
    print(classification_report(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_probs))


--- Naive Bayes ---
              precision    recall  f1-score   support

           0       1.00      0.17      0.28       729
           1       0.06      1.00      0.11        38

    accuracy                           0.21       767
   macro avg       0.53      0.58      0.20       767
weighted avg       0.95      0.21      0.28       767

ROC-AUC: 0.7852862609197893

--- Decision Tree ---
              precision    recall  f1-score   support

           0       0.98      0.74      0.84       729
           1       0.13      0.74      0.22        38

    accuracy                           0.74       767
   macro avg       0.55      0.74      0.53       767
weighted avg       0.94      0.74      0.81       767

ROC-AUC: 0.7433037325824849

--- SVM ---
              precision    recall  f1-score   support

           0       0.98      0.72      0.83       729
           1       0.12      0.74      0.21        38

    accuracy                           0.72       767
   macro avg   

In [13]:
# Thiết lập seed trước Neural Network để đảm bảo reproducible
tf.random.set_seed(42)
np.random.seed(42)
random.seed(42)

nn_model = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu', input_shape=(len(selected_features_idx),)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(1, activation='sigmoid') # Phân loại nhị phân
])

nn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Huấn luyện
history = nn_model.fit(X_train_ga, y_train_res, epochs=50, batch_size=32, validation_data=(X_val_ga, y_val), verbose=1, callbacks=[early_stop])

nn_probs = nn_model.predict(X_test_ga)
threshold = 0.45
# Dự đoán
nn_preds = (nn_probs > threshold).astype(int)
print("\n--- Neural Network ---")
print(classification_report(y_test, nn_preds))
print("ROC-AUC:", roc_auc_score(y_test, nn_probs))

Epoch 1/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - AUC: 0.8166 - loss: 0.5288 - val_AUC: 0.8293 - val_loss: 0.4288
Epoch 2/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8764 - loss: 0.4347 - val_AUC: 0.8269 - val_loss: 0.4387
Epoch 3/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8857 - loss: 0.4150 - val_AUC: 0.8252 - val_loss: 0.4419
Epoch 4/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - AUC: 0.8862 - loss: 0.4158 - val_AUC: 0.8236 - val_loss: 0.4358
Epoch 5/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8905 - loss: 0.4088 - val_AUC: 0.8234 - val_loss: 0.4319
Epoch 6/50
213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - AUC: 0.8932 - loss: 0.4037 - val_AUC: 0.8205 - val_loss: 0.4308
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 

--- Neural Network ---
              precision    recall  f1-score   support

           0       0.98      0.70      0.82       729
           1       0.11      0.71      0.19        38

    accuracy                           0.70       767
   macro a